# Tokenization

# Packages

datasets function might need to be revised, in case this is part of the script uploaded to cluster
--> depends where the data is coming from, might be able to upload somewhere. If uploaded to HF for example, we can't use load_from_disk, but have to use load_datasets

In [ ]:
from datasets import load_from_disk, load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch
from trl import SFTConfig, SFTTrainer
from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model
from huggingface_hub import login, upload_folder
from google.colab import userdata

hf_token = userdata.get("HF-Token")

# local login
#login(token = "HF-Token", add_to_git_credential=True)

#colab login
login(token = hf_token)

In [ ]:
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
print(device)

cuda


In [ ]:
# full dataset local
#raw_datasets = load_from_disk("../data/mbti_dict_ver2")

# colab, get dataset from HF
raw_datasets = load_dataset("DrinkIcedT/mbti")

raw_train_dataset = raw_datasets["train"]
raw_validation_dataset = raw_datasets["validation"]
raw_test_dataset = raw_datasets["test"]


# for running local test, variable dataset size
#raw_datasets = raw_datasets.filter(lambda _, indices: indices % 25 == 0, with_indices=True)
#raw_datasets.save_to_disk("..\data\smol")

# small dataset local
#raw_datasets = load_from_disk("..\data\smol")



# small datasets split
# raw_train_dataset = raw_datasets["train"]
# raw_validation_dataset = raw_datasets["validation"]
# raw_test_dataset = raw_datasets["test"]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


# Checkpoints

In [ ]:
qwen_checkpoint = None
# qwen_checkpoint = "Qwen/Qwen2.5-0.5B-Instruct"
qwen_checkpoint = "Qwen/Qwen2.5-7B-Instruct"
qwen_tokenizer = AutoTokenizer.from_pretrained(qwen_checkpoint)

llama_checkpoint = None
#llama_checkpoint = "meta-llama/Llama-3.1-8B-Instruct"
#llama_tokenizer = AutoTokenizer.from_pretrained(llama_checkpoint)


# Chat-Templates and Tokenization

In [ ]:
#peft config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",      # NormalFloat 4: Speziell für Gewichte optimiert
    bnb_4bit_compute_dtype=torch.float16, # Die Rechenpräzision
    bnb_4bit_use_double_quant=True, # Spart noch ein bisschen mehr VRAM
)

model = AutoModelForCausalLM.from_pretrained(
    qwen_checkpoint,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True, # Oft nötig bei Qwen
    dtype=torch.float16,
)

model = prepare_model_for_kbit_training(model)

rank_dimension = 8
lora_alpha = 16
lora_dropout = 0.05

lora_config = LoraConfig(
    r=rank_dimension,
    lora_alpha=lora_alpha,
    lora_dropout=lora_dropout,
    bias = "none",
    target_modules=["q_proj", "v_proj"],
    task_type="CAUSAL_LM"
)

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

In [ ]:
def convert_to_chatml(example, checkpoint):
    mbti_type = raw_datasets["train"].features["label"].int2str(example["label"])

    if checkpoint == qwen_checkpoint:
        prompt = f"Your personality Type is {mbti_type}. What is on your mind?"
        return {
            "messages": [
                {"role": "user", "content": prompt},
                {"role": "assistant", "content": example["post"]}
            ]
        }
    elif checkpoint == llama_checkpoint:
        return {
            "messages": [
                {"role": "system", "content": f"You are a person with personality type {mbti_type} who responds accordingly!"},
                {"role": "user", "content": "What is on your mind?"},
                {"role": "assistant", "content": example["post"]}
            ]
        }

ds = raw_datasets.map(
    convert_to_chatml,
    fn_kwargs={"checkpoint": qwen_checkpoint}
)

#tokenisierung
def tokenize_function(example):
    # Chat Template anwenden
    text = qwen_tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False
    )
    # Tokenisieren
    tokenized = qwen_tokenizer(
        text,
        truncation=True,
        max_length=512,
        padding=False
    )

    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

#Dataset transformieren
tokenized_ds = ds.map(
    tokenize_function,
    remove_columns=ds["train"].column_names # Löscht 'messages', 'post' etc.
)

#TrainingArgs
training_args = SFTConfig(
    output_dir="./mbti_test_output",
    max_steps=100,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=5e-5,
    logging_steps=10,
    save_steps=100,
    eval_strategy="steps",
    eval_steps=50,
    fp16=False, # Falls der BFloat-Fehler kommt, hier auf False
    bf16=False,
    optim="paged_adamw_32bit",
    gradient_checkpointing=True,
    report_to="none"
)

# trainer
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["validation"],
    peft_config=lora_config,
    # processing_class=qwen_tokenizer # Kann oft weggelassen werden, wenn tokenized
)

# print(ds["train"][0]["messages"])

# messages = ds["train"][0]["messages"]

# text = qwen_tokenizer.apply_chat_template(
#     messages,
#     tokenize = False,
#     add_generation_prompt = True
# )

#model_inputs = qwen_tokenizer([text], return_tensors = "pt")

# training_args = SFTConfig(
#     output_dir = "./mbti_test_output",
#     max_steps = 100,
#     max_length = 512, #oder 256
#     per_device_train_batch_size = 1,
#     gradient_accumulation_steps=4,
#     learning_rate = 5e-5,
#     logging_steps = 10,
#     save_steps=100,
#     eval_strategy = "steps",
#     eval_steps = 50,
#     fp16=True, # Changed to False
#     bf16=False,
#     dataset_kwargs = {"skip_prepare_dataset": True},
#     optim = "paged_adamw_32bit",
#     gradient_checkpointing = True,
#     remove_unused_columns = False,
#     report_to = "none"
# )

# def formatting_prompts_func(example):
#     output_texts = []
#     # Da SFTTrainer die Daten oft "batched" an die Funktion gibt:
#     for i in range(len(example['messages'])):
#         text = qwen_tokenizer.apply_chat_template(
#             example['messages'][i],
#             tokenize=False,
#             add_generation_prompt=False
#         )
#         output_texts.append(text)
#     return output_texts

# trainer = SFTTrainer(
#     model=model,
#     args=training_args,
#     train_dataset=ds["train"],
#     eval_dataset=ds["validation"],
#     processing_class=qwen_tokenizer,
#     peft_config=lora_config,
#     formatting_func=formatting_prompts_func,
# )


Map:   0%|          | 0/296476 [00:00<?, ? examples/s]

Map:   0%|          | 0/37060 [00:00<?, ? examples/s]

Map:   0%|          | 0/37060 [00:00<?, ? examples/s]

Map:   0%|          | 0/296476 [00:00<?, ? examples/s]

Map:   0%|          | 0/37060 [00:00<?, ? examples/s]

Map:   0%|          | 0/37060 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/296476 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/37060 [00:00<?, ? examples/s]

In [ ]:
print(qwen_tokenizer.chat_template)

{%- if tools %}
    {{- '<|im_start|>system\n' }}
    {%- if messages[0]['role'] == 'system' %}
        {{- messages[0]['content'] }}
    {%- else %}
        {{- 'You are Qwen, created by Alibaba Cloud. You are a helpful assistant.' }}
    {%- endif %}
    {{- "\n\n# Tools\n\nYou may call one or more functions to assist with the user query.\n\nYou are provided with function signatures within <tools></tools> XML tags:\n<tools>" }}
    {%- for tool in tools %}
        {{- "\n" }}
        {{- tool | tojson }}
    {%- endfor %}
    {{- "\n</tools>\n\nFor each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:\n<tool_call>\n{\"name\": <function-name>, \"arguments\": <args-json-object>}\n</tool_call><|im_end|>\n" }}
{%- else %}
    {%- if messages[0]['role'] == 'system' %}
        {{- '<|im_start|>system\n' + messages[0]['content'] + '<|im_end|>\n' }}
    {%- else %}
        {{- '<|im_start|>system\nYou are Qwen, created by Alibaba C

In [ ]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss,Validation Loss
50,3.416683,3.274240
100,2.799055,2.850296


TrainOutput(global_step=100, training_loss=3.587188835144043, metrics={'train_runtime': 9789.037, 'train_samples_per_second': 0.041, 'train_steps_per_second': 0.01, 'total_flos': 1395983651371008.0, 'train_loss': 3.587188835144043})

In [ ]:
from transformers import pipeline

# Test prompts
prompt = "Your personality Type is INTJ. What is on your mind?"
# Oder: "Your personality Type is ENFP. What is on your mind?"

pipe = pipeline(task="text-generation", model=model, tokenizer=qwen_tokenizer, max_new_tokens=100)
result = pipe(f"<|im_start|>user\n{prompt}<|im_end|>\n<|im_start|>assistant\n")

print(result[0]['generated_text'])

Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


<|im_start|>user
Your personality Type is INTJ. What is on your mind?<|im_end|>
<|im_start|>assistant
As an INTJ, I tend to be more focused on my thoughts and ideas rather than sharing them with others. However, if you'd like to engage in a thought-provoking discussion or explore some complex concepts, here are a few things that might be on my mind:

1. **Strategic Planning**: I often spend time thinking about long-term plans and strategies for achieving goals. This could involve analyzing current trends, anticipating future changes, and formulating comprehensive plans to stay ahead of the curve


In [ ]:
prompt = "Create a dialog between an INTJ and an ENFP personality type. You play both roles. So it in the format 'Person 1 (INTJ): [...]' and so on. Imagine the two persons are work colleagues."
pipe = pipeline(task="text-generation", model=model, tokenizer=qwen_tokenizer, max_new_tokens=500)
result = pipe(f"<|im_start|>user\n{prompt}<|im_end|>\n<|im_start|>assistant\n")

print(result[0]['generated_text'])

Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


<|im_start|>user
Create a dialog between an INTJ and an ENFP personality type. You play both roles. So it in the format 'Person 1 (INTJ): [...]' and so on. Imagine the two persons are work colleagues.<|im_end|>
<|im_start|>assistant
Person 1 (INTJ): Good morning, Emma. I wanted to discuss the project timeline with you today.

Person 2 (ENFP): Oh, hi Alex! Yeah, sure, what's up? I was just about to start on the presentation for our client meeting next week.

Person 1: Great timing. I've been reviewing the schedule, and we need to be more precise about when each task is due. There's a lot of work that needs to be done, and I want to make sure we stay on track.

Person 2: I see your point. But sometimes it feels like we're micromanaging everything. I think it would be good to have a bit more flexibility in case something comes up.

Person 1: Flexibility is important, but we also need to ensure we meet all deadlines. I believe a structured approach will help us manage the workload better. 

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

# Pfad in deinem Google Drive
save_path = "/content/drive/MyDrive/Masterarbeit_Modelle/mbti_qwen_adapter_100steps"

# Ordner erstellen, falls er noch nicht existiert
os.makedirs(save_path, exist_ok=True)

# Modell (Adapter) und Tokenizer speichern
trainer.model.save_pretrained(save_path)
qwen_tokenizer.save_pretrained(save_path)

print(f"✅ Dein Modell wurde sicher hier gespeichert: {save_path}")

✅ Dein Modell wurde sicher hier gespeichert: /content/drive/MyDrive/Masterarbeit_Modelle/mbti_qwen_adapter_100steps


In [ ]:
import pandas as pd

# Extrahiert die Loss-Werte aus dem Trainer-Log
history = pd.DataFrame(trainer.state.log_history)

# Speichert die Tabelle als CSV in deinem Drive-Ordner
history.to_csv(f"{save_path}/training_history.csv", index=False)

print("Die Loss-Historie wurde als CSV gespeichert!")

Die Loss-Historie wurde als CSV gespeichert!


### Watch out for:
    Watch for these warning signs during training:

        Validation loss increasing while training loss decreases (overfitting)
        No significant improvement in loss values (underfitting)
        Extremely low loss values (potential memorization)
        Inconsistent output formatting (template learning issues)